# SentinelFlow — 04: Agentic Orchestrator

**Stack:** ROCm · vLLM · Qwen2.5 · LangGraph  
**Purpose:** Run the full multi-agent pipeline from `workflow.yml`,  
all LLM calls routed to local Qwen via vLLM.

### Architecture
```
workflow.yml → WorkflowLoader → GraphBuilder → LangGraph StateGraph
                                                      │
              ┌───────────────────────────────────────┤
              │  Orchestrator Agent (Qwen plans next step)
              │  Ingestion Agent   → reads synthetic frames
              │  EdgeFilter Agent  → motion gate + tiny detect
              │  FogCorrelator Agent → Qwen correlation
              │  CloudAnalyser Agent → Qwen deep analysis
              └─ AlertLearn Agent  → route + retrain signal
```
All agents call Qwen via `http://localhost:8000/v1` (OpenAI-compatible).

## 0. Load config

In [ ]:
import yaml, json, pathlib
cfg = yaml.safe_load(pathlib.Path('config.yml').read_text())
VLLM_BASE_URL = cfg['vllm_base_url']
QWEN_MODEL    = 'qwen'

from openai import OpenAI
client = OpenAI(base_url=VLLM_BASE_URL, api_key='not-needed')
print('Config loaded. Qwen client ready.')

## 1. Install LangGraph

In [ ]:
!pip install langgraph langchain-core -q
import langgraph
print(f'LangGraph version: {langgraph.__version__}')

## 2. Shared pipeline state

In [ ]:
from typing import List, Dict, Optional, Any
from typing_extensions import TypedDict

class PipelineState(TypedDict, total=False):
    # Raw data
    raw_frames         : List[Dict]
    sources_active     : List[str]
    # Edge
    detection_events   : List[Dict]
    edge_stats         : Dict
    # Fog
    correlated_events  : List[Dict]
    fog_stats          : Dict
    # Cloud
    cloud_results      : List[Dict]
    cloud_stats        : Dict
    # Alert
    alerts_sent        : List[str]
    notifications      : List[Dict]
    retrain_submitted  : bool
    hitl_pending       : int
    # Orchestration
    plan               : Dict
    completed_nodes    : List[str]
    failed_nodes       : Dict[str, str]
    retry_counts       : Dict[str, int]
    run_id             : str
    latency_log        : List[Dict]

print('PipelineState defined')

## 3. Qwen LLM helper (used by every agent)

In [ ]:
import time, json

def qwen_json(system: str, user: str, max_tokens: int = 512,
              temperature: float = 0.0) -> Dict:
    """
    Call Qwen via vLLM and parse the response as JSON.
    Falls back to an error dict if parsing fails.
    """
    for attempt in range(2):
        suffix = '\n\nIMPORTANT: respond with ONLY valid JSON, no markdown.' if attempt else ''
        try:
            resp = client.chat.completions.create(
                model=QWEN_MODEL,
                messages=[
                    {'role': 'system', 'content': system + suffix},
                    {'role': 'user',   'content': user},
                ],
                max_tokens=max_tokens,
                temperature=temperature,
            )
            raw = resp.choices[0].message.content.strip()
            if raw.startswith('```'):
                raw = '\n'.join(l for l in raw.split('\n')
                                if not l.strip().startswith('```')).strip()
            return json.loads(raw)
        except json.JSONDecodeError:
            if attempt == 0:
                continue
            return {'error': 'json_parse_failed', 'raw': raw[:200]}
        except Exception as e:
            return {'error': str(e)}

print('qwen_json helper ready')

## 4. All six agent nodes

In [ ]:
import uuid, numpy as np
from datetime import datetime

PIPELINE_ORDER = ['ingestion','edge_filter','fog_correlator',
                  'cloud_analyser','alert_learn']

# ── Orchestrator ─────────────────────────────────────────────────────────────
ORCH_SYS = """\
You are the Orchestrator Agent for SentinelFlow (ROCm/Qwen pipeline).
Inspect pipeline state and decide the next step.
Normal order: ingestion → edge_filter → fog_correlator → cloud_analyser → alert_learn → done
On failure: retry up to 2 times, then skip and continue.

Respond ONLY with JSON: {"next": "<node_or_done>", "reason": "<1 sentence>"}
"""

def orchestrator_node(state: PipelineState) -> PipelineState:
    print('\n[orchestrator] planning next step...')
    completed = state.get('completed_nodes', [])
    failed    = state.get('failed_nodes', {})
    retries   = state.get('retry_counts', {})

    # Find next node
    next_node = None
    for node in PIPELINE_ORDER:
        if node not in completed:
            if node in failed and retries.get(node, 0) >= 2:
                print(f'  Skipping failed node: {node}')
                continue
            next_node = node
            break

    if next_node is None:
        state['plan'] = {'next': 'done', 'reason': 'All nodes completed'}
        print('  → done')
        return state

    ctx = (f'Completed: {completed}. Failed: {list(failed)}. '
           f'Suggested next: {next_node}. '
           f'Edge events: {len(state.get("detection_events",[]))}. '
           f'Correlated: {len(state.get("correlated_events",[]))}')
    plan = qwen_json(ORCH_SYS, ctx, max_tokens=128)
    state['plan'] = plan
    print(f'  → {plan.get("next")}  reason: {plan.get("reason","")}')
    return state


# ── Ingestion ─────────────────────────────────────────────────────────────────
INGEST_SYS = """\
You are the Ingestion Agent. Frames have been ingested from cameras and sensors.
Summarise the ingestion result.
Respond ONLY with JSON: {"status": "ok|partial|failed", "message": "<summary>"}
"""

def ingestion_node(state: PipelineState) -> PipelineState:
    print('[ingestion] reading frames...')
    t0 = time.perf_counter()

    # Synthetic frame generation
    rng    = np.random.default_rng(42)
    frames = []
    for cam in ['cam_001', 'cam_002', 'sat_001']:
        for i in range(15):
            scale = 60 if 8 <= i <= 14 else 4
            base  = rng.integers(60, 180, (64,64,3), dtype=np.uint8)
            noise = rng.integers(-scale, scale+1, base.shape)
            data  = np.clip(base.astype(np.int16)+noise, 0, 255).astype(np.uint8)
            frames.append({'source_id': cam, 'frame_idx': i,
                           'data': data.tolist(),
                           'timestamp': datetime.utcnow().isoformat()})

    summary_msg = (f'Ingested {len(frames)} frames from 3 sources '
                   f'(cam_001, cam_002, sat_001).')
    plan = qwen_json(INGEST_SYS, summary_msg, max_tokens=128)

    ms = (time.perf_counter() - t0)*1000
    state['raw_frames']      = frames
    state['sources_active']  = ['cam_001','cam_002','sat_001']
    completed = list(state.get('completed_nodes', []))
    completed.append('ingestion')
    state['completed_nodes'] = completed
    log = list(state.get('latency_log', []))
    log.append({'node':'ingestion','ms':round(ms,1)})
    state['latency_log'] = log
    print(f'  ✓ {len(frames)} frames  {ms:.0f}ms  status={plan.get("status")}')
    return state


# ── Edge Filter ───────────────────────────────────────────────────────────────
EDGE_SYS = """\
You are the Edge Filter Agent. You receive edge filtering statistics.
Summarise what was filtered and why. Mention data reduction achieved.
Respond ONLY with JSON:
{"frames_processed": N, "events_emitted": N,
 "data_reduction_pct": N, "message": "<summary>"}
"""

def edge_filter_node(state: PipelineState) -> PipelineState:
    print('[edge_filter] filtering frames...')
    t0     = time.perf_counter()
    frames = state.get('raw_frames', [])

    LABELS = ['person','vehicle','bicycle','unknown_object','ship','wildfire_smoke']
    rng    = np.random.default_rng(7)
    events = []
    prev   = None

    for f in frames:
        data = np.array(f['data'], dtype=np.uint8)
        gray = data.mean(axis=-1)
        # Motion gate
        if prev is not None:
            diff  = np.abs(gray.astype(float) - prev.astype(float))
            score = float((diff > 10).mean())
            if score < 0.015:
                prev = gray; continue
        prev = gray
        # Tiny detect
        seed = int(data.flatten()[:4].sum()) % 9999
        r2   = np.random.default_rng(seed)
        for _ in range(int(r2.integers(0,3))):
            conf = float(r2.uniform(0.35, 0.98))
            if conf < 0.45: continue
            events.append({
                'event_id'  : str(uuid.uuid4()),
                'source_id' : f['source_id'],
                'timestamp' : f['timestamp'],
                'label'     : str(r2.choice(LABELS)),
                'confidence': round(conf, 4),
                'bbox'      : {'x':float(r2.uniform(0.05,0.75)),
                               'y':float(r2.uniform(0.05,0.75)),
                               'w':float(r2.uniform(0.05,0.25)),
                               'h':float(r2.uniform(0.05,0.25))},
            })

    red = (1 - len(events)/max(len(frames),1))*100
    msg = (f'Processed {len(frames)} frames → {len(events)} events '
           f'({red:.0f}% reduction).')
    plan = qwen_json(EDGE_SYS, msg, max_tokens=192)

    ms = (time.perf_counter() - t0)*1000
    state['detection_events'] = events
    state['edge_stats'] = plan
    completed = list(state.get('completed_nodes',[]))
    completed.append('edge_filter')
    state['completed_nodes'] = completed
    log = list(state.get('latency_log',[]))
    log.append({'node':'edge_filter','ms':round(ms,1)})
    state['latency_log'] = log
    print(f'  ✓ {len(events)} events  {ms:.0f}ms  reduction={red:.0f}%')
    return state


# ── Fog Correlator ────────────────────────────────────────────────────────────
FOG_SYS = """\
You are the Fog Correlator Agent. Analyse detection events and identify correlations.
Apply rules: 2+ cams with same label → escalate. Unknown + high conf → escalate.
Respond ONLY with JSON:
{
  "correlated_events": [
    {"correlation_id": "<uuid>", "event_type": "<name>",
     "severity": "low|medium|high|critical",
     "involved_sources": ["<id>"],
     "description": "<str>", "should_escalate": true,
     "avg_confidence": 0.0, "rule_matched": "<str>"}
  ],
  "escalated_count": N, "message": "<summary>"
}
"""

def fog_correlator_node(state: PipelineState) -> PipelineState:
    print('[fog_correlator] correlating events...')
    t0     = time.perf_counter()
    events = state.get('detection_events', [])

    from collections import defaultdict
    by_lbl = defaultdict(list)
    for e in events:
        by_lbl[e['label']].append(e)

    summary = [
        {'label': lbl,
         'count': len(evs),
         'sources': list({e['source_id'] for e in evs}),
         'avg_conf': round(sum(e['confidence'] for e in evs)/len(evs),3)}
        for lbl, evs in by_lbl.items()
    ]
    prompt = f'Window events summary: {json.dumps(summary)}. Correlate and escalate.'
    plan   = qwen_json(FOG_SYS, prompt, max_tokens=512)

    # Ensure UUIDs
    for c in plan.get('correlated_events', []):
        if not c.get('correlation_id'):
            c['correlation_id'] = str(uuid.uuid4())
        c['source_events'] = [
            e for e in events
            if e.get('source_id') in c.get('involved_sources', [])
        ]

    ms = (time.perf_counter() - t0)*1000
    state['correlated_events'] = plan.get('correlated_events', [])
    state['fog_stats'] = plan
    completed = list(state.get('completed_nodes',[]))
    completed.append('fog_correlator')
    state['completed_nodes'] = completed
    log = list(state.get('latency_log',[]))
    log.append({'node':'fog_correlator','ms':round(ms,1)})
    state['latency_log'] = log
    n = plan.get('escalated_count', len(plan.get('correlated_events',[])))
    print(f'  ✓ correlated={len(plan.get("correlated_events",[]))} '
          f'escalated={n}  {ms:.0f}ms')
    return state


# ── Cloud Analyser ────────────────────────────────────────────────────────────
CLOUD_SYS = """\
You are the Cloud Analyser Agent. Perform deep analysis on each escalated event.
Respond ONLY with JSON:
{
  "results": [
    {"analysis_id": "<uuid>", "correlation_id": "<id>",
     "anomaly_score": 0.0,
     "summary": "<2-3 sentence narrative>",
     "recommended_action": "<actionable>",
     "retrain_flag": false}
  ],
  "avg_anomaly_score": 0.0,
  "retrain_triggered": false,
  "message": "<summary>"
}
"""

def cloud_analyser_node(state: PipelineState) -> PipelineState:
    print('[cloud_analyser] deep analysis...')
    t0   = time.perf_counter()
    evts = [e for e in state.get('correlated_events',[]) if e.get('should_escalate')]

    if not evts:
        print('  No escalated events — skipping')
        state['cloud_results'] = []
        completed = list(state.get('completed_nodes',[]))
        completed.append('cloud_analyser')
        state['completed_nodes'] = completed
        return state

    prompt = f'Escalated events ({len(evts)}): {json.dumps(evts, default=str)[:2000]}'
    plan   = qwen_json(CLOUD_SYS, prompt, max_tokens=1024)

    for r in plan.get('results', []):
        if not r.get('analysis_id'):
            r['analysis_id'] = str(uuid.uuid4())

    ms = (time.perf_counter() - t0)*1000
    state['cloud_results'] = plan.get('results', [])
    state['cloud_stats']   = plan
    completed = list(state.get('completed_nodes',[]))
    completed.append('cloud_analyser')
    state['completed_nodes'] = completed
    log = list(state.get('latency_log',[]))
    log.append({'node':'cloud_analyser','ms':round(ms,1)})
    state['latency_log'] = log
    avg = plan.get('avg_anomaly_score', 0)
    print(f'  ✓ {len(plan.get("results",[]))} results  avg_anomaly={avg:.3f}  {ms:.0f}ms')
    return state


# ── Alert & Learn ─────────────────────────────────────────────────────────────
ALERT_SYS = """\
You are the Alert & Learn Agent. Route analysis results to the right channel.
Critical → webhook. High → Slack. Medium → dashboard. Low → log only.
Respond ONLY with JSON:
{
  "alerts_sent": ["<analysis_id>"],
  "notifications": [{"channel": "<ch>", "severity": "<sev>", "message": "<msg>"}],
  "retrain_submitted": false,
  "hitl_pending": N,
  "message": "<summary>"
}
"""

def alert_learn_node(state: PipelineState) -> PipelineState:
    print('[alert_learn] routing alerts...')
    t0      = time.perf_counter()
    results = state.get('cloud_results', [])
    events  = state.get('correlated_events', [])

    prompt = (
        f'Analysis results ({len(results)}): {json.dumps(results, default=str)[:1500]}\n'
        f'Correlated events severities: '
        f'{[e.get("severity") for e in events]}\n'
        f'Retrain triggered: {state.get("cloud_stats",{}).get("retrain_triggered",False)}'
    )
    plan = qwen_json(ALERT_SYS, prompt, max_tokens=512)

    ms = (time.perf_counter() - t0)*1000
    state['alerts_sent']    = plan.get('alerts_sent', [])
    state['notifications']  = plan.get('notifications', [])
    state['retrain_submitted'] = plan.get('retrain_submitted', False)
    state['hitl_pending']   = plan.get('hitl_pending', 0)
    completed = list(state.get('completed_nodes',[]))
    completed.append('alert_learn')
    state['completed_nodes'] = completed
    log = list(state.get('latency_log',[]))
    log.append({'node':'alert_learn','ms':round(ms,1)})
    state['latency_log'] = log
    print(f'  ✓ {len(plan.get("notifications",[]))} notifications  '
          f'retrain={plan.get("retrain_submitted")}  {ms:.0f}ms')
    for n in plan.get('notifications', []):
        icon = {'webhook':'🔴','slack':'🟠','dashboard':'🟡','log':'🔵'}.get(
            n.get('channel',''),'⚪')
        print(f'    {icon} [{n.get("severity","").upper():<8}] '
              f'{n.get("channel")}: {n.get("message","")[:80]}')
    return state

print('All 6 agent nodes defined')

## 5. Load workflow.yml + build graph

In [ ]:
from langgraph.graph import StateGraph, END

# Load topology from workflow.yml
wf = yaml.safe_load(pathlib.Path('workflow.yml').read_text())

NODE_FNS = {
    'orchestrator'  : orchestrator_node,
    'ingestion'     : ingestion_node,
    'edge_filter'   : edge_filter_node,
    'fog_correlator': fog_correlator_node,
    'cloud_analyser': cloud_analyser_node,
    'alert_learn'   : alert_learn_node,
}

graph = StateGraph(PipelineState)

# ── Add nodes from workflow.yml ──────────────────────────────────────────────
for node_name in wf['nodes']:
    if node_name in NODE_FNS:
        graph.add_node(node_name, NODE_FNS[node_name])
        print(f'  + node: {node_name}')
    else:
        print(f'  ⚠ node {node_name} has no Python function — skipped')

# ── Entry point from workflow.yml ────────────────────────────────────────────
graph.set_entry_point(wf['entry_point'])
print(f'  entry_point: {wf["entry_point"]}')

# ── Routing: read from workflow.yml routing section ──────────────────────────
def make_router(source_field: str, routes: Dict, fallback: str):
    def router(state: PipelineState) -> str:
        val  = state.get('plan', {}).get(source_field, '')
        dest = routes.get(val, fallback)
        return END if dest == '__end__' else dest
    return router

for node_name, rc in wf['routing'].items():
    if node_name not in NODE_FNS:
        continue
    if rc['type'] == 'conditional':
        routes  = rc['routes']
        fallback= rc['fallback']
        fn      = make_router(rc['source_field'], routes, fallback)
        path_map = {}
        for _, dest in routes.items():
            real = END if dest == '__end__' else dest
            path_map[real] = real
        fb_real = END if fallback == '__end__' else fallback
        path_map[fb_real] = fb_real
        graph.add_conditional_edges(node_name, fn, path_map)
        print(f'  + conditional edges: {node_name} → {list(path_map.keys())}')
    elif rc['type'] == 'fixed':
        dest = rc.get('to', 'orchestrator')
        real = END if dest == '__end__' else dest
        if node_name in NODE_FNS and (dest in NODE_FNS or dest == '__end__'):
            graph.add_edge(node_name, real)
            print(f'  + fixed edge: {node_name} → {real}')

app = graph.compile()
print('\n✓ LangGraph compiled from workflow.yml')

## 6. Run the full agentic pipeline

In [ ]:
run_id = str(uuid.uuid4())[:8]
print(f'Starting SentinelFlow agentic run  [run_id={run_id}]')
print('═' * 60)

initial: PipelineState = {
    'raw_frames'        : [],
    'detection_events'  : [],
    'correlated_events' : [],
    'cloud_results'     : [],
    'alerts_sent'       : [],
    'notifications'     : [],
    'plan'              : {},
    'completed_nodes'   : [],
    'failed_nodes'      : {},
    'retry_counts'      : {},
    'run_id'            : run_id,
    'latency_log'       : [],
    'retrain_submitted' : False,
    'hitl_pending'      : 0,
}

t_total = time.perf_counter()
final   = app.invoke(initial)
total_ms= (time.perf_counter() - t_total) * 1000

print('\n' + '═'*60)
print('  PIPELINE COMPLETE')
print('═'*60)

## 7. Results & latency report

In [ ]:
print(f'\nRun ID          : {final["run_id"]}')
print(f'Total wall time : {total_ms/1000:.1f}s')
print(f'Completed nodes : {final["completed_nodes"]}')
if final.get('failed_nodes'):
    print(f'Failed nodes    : {final["failed_nodes"]}')

print(f'\nData volumes:')
print(f'  Raw frames         : {len(final["raw_frames"])}')
print(f'  Detection events   : {len(final["detection_events"])}')
print(f'  Correlated events  : {len(final["correlated_events"])}')
print(f'  Cloud results      : {len(final["cloud_results"])}')
print(f'  Alerts sent        : {len(final["alerts_sent"])}')
print(f'  HITL pending       : {final.get("hitl_pending",0)}')
print(f'  Retrain submitted  : {final.get("retrain_submitted")}')

print(f'\nLatency per node:')
for entry in final.get('latency_log', []):
    budget = {'ingestion':200,'edge_filter':50,'fog_correlator':500,
              'cloud_analyser':5000,'alert_learn':500}.get(entry['node'],9999)
    flag = '⚠' if entry['ms'] > budget else '✓'
    print(f'  {flag} {entry["node"]:<20} {entry["ms"]:>8.1f} ms  '
          f'(budget {budget} ms)')

print(f'\nNotifications:')
for n in final.get('notifications', []):
    icon = {'webhook':'🔴','slack':'🟠','dashboard':'🟡','log':'🔵'}.get(
        n.get('channel',''),'⚪')
    print(f'  {icon} [{n.get("severity","").upper():<8}] '
          f'{n.get("channel")}: {n.get("message","")[:90]}')

## 8. Save final state

In [ ]:
# Remove raw numpy arrays before serialising
save_state = {k: v for k, v in final.items() if k != 'raw_frames'}
pathlib.Path('final_pipeline_state.json').write_text(
    json.dumps(save_state, default=str, indent=2)
)
print('Saved final_pipeline_state.json')
print('\n✓ SentinelFlow — all notebooks complete')